In [5]:
import sys
from pathlib import Path
from dotenv import load_dotenv
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
    filter_financial_news_published_today,
    get_financial_news_content_by_id,
    normalize_financial_news_datetime_column,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import date, datetime

# Sanity check: if this fails, use File → Reload Notebook from Disk, then restart kernel
import storage.postgres.news_dataframe as _news_df
print(f"Using news_dataframe from: {_news_df.__file__}")
print(f"filter_financial_news_ingested_today: OK")

Using news_dataframe from: /app/src/storage/postgres/news_dataframe.py
filter_financial_news_ingested_today: OK


In [6]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [7]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 290


In [8]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,3225040353072462344,24/7 Wall St.,"$25,000 in XRP vs $25,000 in S&P 500: Backtest...",https://finance.yahoo.com/markets/crypto/artic...,,"Quick Read\nA $25,000 investment in XRP at the...",Sam Daodu,6 min read,2026-05-22 16:02:26,2026-05-23 03:23:41.886219
1,4446138805452953510,24/7 Wall St.,"$5,000 in XRP at $1.37 vs Bitcoin at $77,000: ...",https://finance.yahoo.com/markets/crypto/artic...,,"Quick Read\nInvesting $5,000 at $1.37 buys 3,6...",Sam Daodu,8 min read,2026-05-22 08:36:29,2026-05-23 03:23:42.148369
2,1910953675637808579,BeInCrypto,$725 Million in Ethereum (ETH) Just Left Whale...,https://finance.yahoo.com/markets/crypto/artic...,,"Ethereum (ETH) price trades at $2,132 on May 2...",Ananda Banerjee,3 min read,2026-05-22 07:00:35,2026-05-23 03:23:42.185079
3,3378378421304511398,BeInCrypto,1 Quadrillion MAPO Minted: Bridge Exploit Cras...,https://finance.yahoo.com/markets/crypto/artic...,,MAP Protocol’s Butter Bridge suffered a severe...,Lockridge Okoth,2 min read,2026-05-20 17:03:35,2026-05-23 03:23:43.236435
4,2570045032087381483,BeInCrypto,10 Surprising Facts About Elon Musk’s $1 Trill...,https://finance.yahoo.com/markets/stocks/artic...,,Elon Musk's SpaceX filed for an initial public...,Mohammad Shahid,3 min read,2026-05-21 20:59:49,2026-05-23 03:23:42.431488


In [9]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_by_date(df)

    # Extract date strings (DB/delete API expects stored string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [10]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, hour=None, minute=None, second=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
                second=second,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Article publish date (datetime) on today's calendar date (default on_date=None)
            return filter_financial_news_by_date(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [11]:
etl = DataETL(df)

# Export by article publish time (datetime). on_date=None → today's local date (date.today()).
export_on_date = "2026-05-24"  # e.g. "2026-05-23" to override today
filtered_df = filter_financial_news_by_date(df, on_date=export_on_date)

print(
    f"Filter date (datetime column): {export_on_date or date.today()} | "
    f"Rows matched: {len(filtered_df)} | "
    f"Ingested today (created_at only): {len(filter_financial_news_ingested_today(df))}"
)
filtered_df.head()

Filter date (datetime column): 2026-05-24 | Rows matched: 24 | Ingested today (created_at only): 24


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
6,1221589746717124508,Motley Fool,3 Cryptocurrencies to Watch as the Clarity Act...,https://finance.yahoo.com/markets/crypto/artic...,,Crypto investors are about to get a lot more c...,"Dominic Basulto, The Motley Fool",3 min read,2026-05-24 04:50:00,2026-05-25 03:14:31.113889
9,2650580303084354710,BeInCrypto,A Bitcoin Treasury Company Has a Doctor on Sta...,https://finance.yahoo.com/markets/crypto/artic...,,Nakamoto Inc. (NAKA) has defended why a Bitcoi...,Lockridge Okoth,2 min read,2026-05-24 21:28:51,2026-05-25 03:08:33.029652
10,1460905467489013234,BeInCrypto,AI Cost Crisis Emerges as Claude Usage and Age...,https://finance.yahoo.com/sectors/technology/a...,,Enterprise AI spending is outrunning corporate...,Lockridge Okoth,2 min read,2026-05-24 19:56:12,2026-05-25 03:14:30.980534
45,4371302684767048796,BeInCrypto,"Buy, Hodl, Repeat: Adam Back Delivers a Clear ...",https://finance.yahoo.com/markets/crypto/artic...,,Blockstream CEO Adam Back says efficient marke...,Phil Haunhorst,2 min read,2026-05-24 15:46:53,2026-05-25 03:14:31.036756
47,3972469859353210199,BeInCrypto,CZ “Surfing Accident” Hoax Sparks Meme Coin Fr...,https://finance.yahoo.com/markets/crypto/artic...,,Changpeng Zhao (CZ) denied a viral rumor that ...,Lockridge Okoth,2 min read,2026-05-24 20:43:49,2026-05-25 03:14:30.959968


In [8]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 24


In [9]:
import os

export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"

if export_rows_to_s3 and not filtered_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Uncomment and set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter: ./docker/start_jupyter.ps1"
        )
    export_df = normalize_financial_news_datetime_column(filtered_df)
    etl_export.set_dataframe(export_df)
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' already exists.
Data for row 6 with id '1221589746717124508' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=05/day=24/hour=04/minute=50/second=00/format=csv/1221589746717124508.csv'
Data for row 9 with id '2650580303084354710' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=05/day=24/hour=21/minute=28/second=51/format=csv/2650580303084354710.csv'
Data for row 10 with id '1460905467489013234' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=05/day=24/hour=19/minute=56/second=12/format=csv/1460905467489013234.csv'
Data for row 45 with id '4371302684767048796' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=05/day=24/hour=15/minute=46/second=53/format=csv/4371302684767048796.csv'
Data for row 47 with id '3972469859353210199' uploaded to S3 bucket 'test-financial-news-bucket' u

In [10]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [12]:
ingest_data = True
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2026'
    month = '05'
    day = '24'
    hour = None   # set e.g. '04' to narrow to one hour; None = whole day
    minute = None
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

Listing s3://test-financial-news-bucket/news/crypto/year=2026/month=05/day=24/
Found 24 CSV file(s) under prefix


In [ ]:
if df_from_file is not None:
    print(df_from_file.count())
    df_from_file.head()

id            24
source        24
headline      24
href          24
summary        0
content       24
author        24
minsread      24
datetime      24
created_at    24
dtype: int64


In [ ]:
targetId = ""  # i.e. "1221589746717124508" targetId can be string or numeric type

# Lookup order: S3 ingest result, filtered export batch, then full DB pull
lookup_df = None
lookup_source = None
for name, candidate in (
    ("df_from_file", df_from_file if "df_from_file" in dir() else None),
    ("filtered_df", filtered_df if "filtered_df" in dir() else None),
    ("df", df if "df" in dir() else None),
):
    if candidate is not None and not getattr(candidate, "empty", True):
        lookup_df = candidate
        lookup_source = name
        break

if targetId and lookup_df is not None:
    full_content = get_financial_news_content_by_id(lookup_df, targetId)
    if full_content:
        print(full_content)
    else:
        print(
            f"No content for id {targetId!r} in {lookup_source} "
            f"({len(lookup_df)} rows). Id column dtype: {lookup_df['id'].dtype}"
        )
else:
    print("Set targetId and ensure df_from_file, filtered_df, or df is loaded.")

[]


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,1221589746717124508,Motley Fool,3 Cryptocurrencies to Watch as the Clarity Act...,https://finance.yahoo.com/markets/crypto/artic...,NaN,Crypto investors are about to get a lot more c...,"Dominic Basulto, The Motley Fool",3 min read,2026-05-24T04:50:00.000Z,2026-05-25 03:14:31.113889
1,4328894494074471180,BeInCrypto,"ECB Pushes Back on Euro Stablecoin Proposals, ...",https://finance.yahoo.com/economy/policy/artic...,NaN,The European Central Bank has warned EU financ...,Phil Haunhorst,2 min read,2026-05-24T08:00:00.000Z,2026-05-25 03:14:31.106868
2,50611771794356397,BeInCrypto,StablR Stablecoins Depeg After $2.8 Million Ex...,https://finance.yahoo.com/markets/crypto/artic...,NaN,StablR's Euro (EURR) and StablR USD (USDR) sta...,Phil Haunhorst,2 min read,2026-05-24T08:50:28.000Z,2026-05-25 03:14:31.099955
3,1256733553333549900,Motley Fool,Could Buying XRP Today Set You Up for Life?,https://finance.yahoo.com/markets/crypto/artic...,NaN,XRP (CRYPTO: XRP) is dangerously close to dipp...,"Dominic Basulto, The Motley Fool",4 min read,2026-05-24T09:28:00.000Z,2026-05-25 03:14:31.093086
4,2527753283082034004,Motley Fool,Sell in May and Go Away: 3 Cryptocurrencies to...,https://finance.yahoo.com/markets/crypto/artic...,NaN,"On Wall Street, there's a famous adage, ""Sell ...","Dominic Basulto, The Motley Fool",4 min read,2026-05-24T09:51:00.000Z,2026-05-25 03:14:31.086592
